In [1]:
from collections import defaultdict
import math

import pickle

with open("data_splits.pkl", "rb") as f:
    train, val, test, vocab = pickle.load(f)



In [2]:
# Count trigrams and bigram contexts
trigram_counts = defaultdict(int)
bigram_counts = defaultdict(int)

for sentence in train:
    for i in range(len(sentence) - 2):
        w1, w2, w3 = sentence[i], sentence[i+1], sentence[i+2]
        trigram_counts[(w1, w2, w3)] += 1
        bigram_counts[(w1, w2)] += 1


In [3]:
def trigram_prob(w1, w2, w3, k=1.0):
    vocab_size = len(vocab)
    numerator = trigram_counts[(w1, w2, w3)] + k
    denominator = bigram_counts[(w1, w2)] + k * vocab_size
    return numerator / denominator


In [4]:
def compute_perplexity(data):
    log_prob_sum = 0
    token_count = 0

    for sentence in data:
        for i in range(len(sentence) - 2):
            w1, w2, w3 = sentence[i], sentence[i+1], sentence[i+2]
            prob = trigram_prob(w1, w2, w3)
            log_prob_sum += math.log(prob)
            token_count += 1

    return math.exp(-log_prob_sum / token_count)


In [5]:
test_perplexity = compute_perplexity(test)
print("Trigram Test Perplexity:", test_perplexity)


Trigram Test Perplexity: 2715.009739190652


In [6]:
import random

def generate_text(max_len=30):
    w1, w2 = "<bos>", "<bos>"
    result = []

    for _ in range(max_len):
        candidates = []
        probs = []

        for w in vocab:
            p = trigram_prob(w1, w2, w)
            candidates.append(w)
            probs.append(p)

        w3 = random.choices(candidates, weights=probs)[0]

        if w3 == "<eos>":
            break

        result.append(w3)
        w1, w2 = w2, w3

    return " ".join(result)


In [7]:
for i in range(3):
    print(f"Sample {i+1}:")
    print(generate_text(30))
    print()


Sample 1:
keenest ceased adapted servant formal number right bordered duplicity disposition study amiable honour drawing assert comprehend mere felicity laughed papa plans desired veracity expences arising fathers declared heavy earl borrow

Sample 2:
pleasing dirt finally composedly made ladyship owing readiness tomorrow affording changes unassuming direct involving spare judging governed drily hither door glad dispatched perplexity handsomest mark situation letter gain silent false

Sample 3:
players injure tempted expressions become successful printed given disadvantage ingenious repeated sparkled fell expences discharging attractions proceeded wants cherished affirmative mr walker indifference dining ugly loveliness insensibility think argument attraction

